In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

train = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/train.csv')
test = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/test.csv')
sample = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/Sample.csv')


In [2]:

train['is_train'] = 1
test['is_train'] = 0

test_temp = test.copy()
test_temp['label'] = np.nan 

all_data = pd.concat([train, test_temp], axis=0, ignore_index=True)

thread_stats = all_data.groupby('post_id').agg({
    'comment': 'count',              # How long is the thread? (Long threads = arguments)
    'upvote': ['mean', 'std', 'sum'], # Is the thread popular?
    'downvote': ['mean', 'std', 'sum'] # Is the thread hated/controversial?
})

thread_stats.columns = [
    'thread_vol', 
    'thread_up_mean', 'thread_up_std', 'thread_up_sum', 
    'thread_down_mean', 'thread_down_std', 'thread_down_sum'
]
thread_stats = thread_stats.reset_index()

thread_stats['thread_engagement'] = thread_stats['thread_up_sum'] + thread_stats['thread_down_sum']
thread_stats['thread_negativity_ratio'] = thread_stats['thread_down_sum'] / (thread_stats['thread_up_sum'] + 1)

train = train.merge(thread_stats, on='post_id', how='left')
test = test.merge(thread_stats, on='post_id', how='left')

fill_zero_cols = ['thread_up_std', 'thread_down_std']
train[fill_zero_cols] = train[fill_zero_cols].fillna(0)
test[fill_zero_cols] = test[fill_zero_cols].fillna(0)

print("Thread features added successfully.")

Thread features added successfully.


In [3]:
import re

def add_term_indicator_columns(
    df,
    terms,
    prefix,
    case_insensitive=True
):
    df = df.copy()

    flags = re.IGNORECASE if case_insensitive else 0

    for term in terms:
        col_name = f"{prefix}_{term}"
        pattern = re.escape(term)
        df[col_name] = df["comment"].str.contains(
            pattern,
            regex=True,
            flags=flags,
            na=False
        )

    return df


In [4]:
# === Feature Engineering and Preprocessing ===
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import FeatureUnion


thread_features = [
    'thread_vol', 'thread_up_mean', 'thread_up_std', 
    'thread_down_mean', 'thread_down_std', 
    'thread_engagement', 'thread_negativity_ratio'
]

# Update X_full to include these new columns
# Note: We added 'post_id' to the list just in case, but we won't use it in the model directly
cols_to_use = [
    'emoticon_1', 'emoticon_2', 'emoticon_3', 
    'upvote', 'downvote', "if_1", "if_2", 
    "disability", "race", "religion", "gender", 
    'comment'
] + thread_features 

X_full = train[cols_to_use].copy()
y = train['label']

X_full["text_len"] = X_full["comment"].str.len()
X_full["word_count"] = X_full["comment"].str.split().str.len()

X_full["question_marks"] = X_full["comment"].str.count(r"\?")
X_full["exclamations"] = X_full["comment"].str.count(r"!")

X_full[["text_len", "word_count"]]  = X_full[["text_len", "word_count"]].fillna(0)
X_full[["question_marks", "exclamations",]] = X_full[["question_marks", "exclamations"]].fillna(False)

X_full = X_full.fillna('Null')

violent_terms = [
    "kill", "die", "death", "shoot", "fire", "killing", 
    "dead", "hang", "hit", "burn", "murder", "rid",
    "shot", "punch", "shooting", "guns", "suicide", "bomb",
    "gun", "fired", "kick", "knife", "killed", "torture",
    "bullet", "slap", "destroy", "stab"
]
X_full = add_term_indicator_columns(X_full,terms=violent_terms,prefix="v")

moral_terms = ["deserve", "should", "solution", "need to", "have to", "all of them"]
X_full = add_term_indicator_columns(X_full,terms=moral_terms,prefix="m")

group_terms = ["muslim", "jew", "jewish", "christian", "catholic", "women", "men", "female", "male",
                "black", "white", "asian", "immigrant", "refugee", "gay", "lesbian", "native", "indian"]

X_full = add_term_indicator_columns(X_full,terms=group_terms,prefix="g")

personal_attack = ["you", "your", "idiot", "stupid", "blind", "troll", "liar", "dumb", "moron"]

X_full = add_term_indicator_columns(X_full,terms=personal_attack,prefix="p")

base_num_features = [
    'emoticon_1', 'emoticon_2', 'emoticon_3', 'upvote', 'downvote', "if_1", "if_2", 
    "disability", "text_len", "word_count", "question_marks", "exclamations"
    ] + thread_features

term_feature_cols = [
    c for c in X_full.columns
    if c.startswith(("v_", "m_", "g_", "p_"))
]

num_features = base_num_features + term_feature_cols
text_feature = "comment"
cat_features = ["race", "religion", "gender"]


feature_encoder = ColumnTransformer(
    transformers=[
        ("text_vec", TfidfVectorizer(max_features=20000,ngram_range=(1, 2)), text_feature),
        ("num_scaler", StandardScaler(), num_features), 
        ("cats", OneHotEncoder(handle_unknown="ignore"), cat_features)
    ],
)

In [5]:
import lightgbm as lgb
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
import time


# Train/validation split - stratified for balanced classes
X_train, X_valid, y_train, y_valid = train_test_split(
    X_full, y, test_size=0.2, random_state=0, stratify=y
)


modelA = Pipeline([
        ("prep", feature_encoder),
        ("model", lgb.LGBMClassifier(
        n_estimators=600,
        learning_rate=0.05,
        num_leaves=64,
        max_depth=10,
        min_child_samples=30,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.5,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=0,
        verbosity=-1
        ))
    ])

modelA.fit(X_train,y_train)

print(classification_report(y_valid, modelA.predict(X_valid)))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


              precision    recall  f1-score   support

           0       0.98      0.95      0.96     22835
           1       0.79      0.77      0.78      3183
           2       0.84      0.92      0.88     12488
           3       0.75      0.48      0.58      1094

    accuracy                           0.91     39600
   macro avg       0.84      0.78      0.80     39600
weighted avg       0.91      0.91      0.91     39600



In [26]:
from scipy.optimize import minimize
from sklearn.metrics import f1_score, classification_report
import numpy as np

# ==========================================
# PHASE 4 (HYBRID): AGGRESSIVE OPTIMIZATION
# ==========================================

print("Generating validation probabilities...")
val_probs = modelA.predict_proba(X_valid) 

# 1. Define the aggressive stance
# A threshold of 0.2 roughly implies we weight Class 3 about 5x more than others
# (Because normally argmax is > 0.25 in 4-class, so 0.25/0.2 ≈ 1.25, but effectively higher)
# Let's start the optimizer with your manual success as the baseline.
manual_weights = np.array([1.0, 1.0, 1.0, 2.0]) # Start Class 3 high

def get_hybrid_f1(oof_probs, y_true):
    """
    Optimizes weights for Classes 0, 1, 2 but keeps Class 3 aggressive.
    """
    def f1_loss(weights):
        # We optimize 3 weights, but we append the fixed Class 3 weight
        # or we optimize all 4 but start from a high point
        
        # Strategy: Optimize all 4, but bound Class 3 to be HIGH
        w = np.maximum(weights, 0) 
        weighted_probs = oof_probs * w
        final_preds = np.argmax(weighted_probs, axis=1)
        return -f1_score(y_true, final_preds, average='macro')

    # Start with your manual "sweet spot" equivalent
    # If threshold 0.2 worked, that's like saying "If prob is 0.2, treat as 1.0"
    # So weight ~ 5.0. Let's try starting there.
    init_weights = np.array([1.0, 1.0, 1.0, 3.5]) 
    
    # We use 'Powell' method here as it's often better at finding 
    # specific peaks in discrete metrics than Nelder-Mead
    opt_result = minimize(
        f1_loss, 
        init_weights, 
        method='Powell', 
        tol=1e-5,
        options={'maxiter': 300}
    )
    
    return opt_result.x, -opt_result.fun

# --- EXECUTION ---

# 1. Run Hybrid Optimization
print("\nRunning Hybrid Optimization (Starting with Aggressive Class 3)...")
best_weights, optimized_f1 = get_hybrid_f1(val_probs, y_valid)

print("-" * 30)
print(f"Optimized Macro F1: {optimized_f1:.5f}")
print(f"Best Class Weights: {np.round(best_weights, 3)}")
print("-" * 30)

# 2. Compare with your Manual Threshold
# Your manual method:
manual_preds = np.argmax(val_probs, axis=1)
manual_preds[val_probs[:, 3] > 0.2] = 3
manual_f1 = f1_score(y_valid, manual_preds, average='macro')
print(f"Your Manual Threshold F1: {manual_f1:.5f}")

if optimized_f1 > manual_f1:
    print("✅ Optimization Found a Better Strategy!")
    final_weights = best_weights
else:
    print("⚠️ Manual Threshold is still better. Using Manual Logic.")
    # If manual is better, we approximate it with weights for submission consistency
    final_weights = np.array([1.0, 1.0, 1.0, 4.0]) # Heuristic equivalent

Generating validation probabilities...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Running Hybrid Optimization (Starting with Aggressive Class 3)...
------------------------------
Optimized Macro F1: 0.81572
Best Class Weights: [0.779 1.586 0.962 3.573]
------------------------------
Your Manual Threshold F1: 0.81252
✅ Optimization Found a Better Strategy!


In [24]:
import numpy as np

def predict_with_threshold(model, X, class_idx=3, threshold=0.2):
    probs = model.predict_proba(X)
    y_pred = np.argmax(probs, axis=1)
    y_pred[probs[:, class_idx] > threshold] = class_idx
    return y_pred


In [25]:
y_pred = predict_with_threshold(modelA, X_valid, class_idx=3, threshold=0.2)

print(classification_report(y_valid, y_pred))
print("Macro F1:", f1_score(y_valid, y_pred, average="macro"))
#LGBM SCORE

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


              precision    recall  f1-score   support

           0       0.98      0.95      0.96     22835
           1       0.79      0.76      0.78      3183
           2       0.85      0.90      0.88     12488
           3       0.61      0.66      0.63      1094

    accuracy                           0.91     39600
   macro avg       0.81      0.82      0.81     39600
weighted avg       0.91      0.91      0.91     39600

Macro F1: 0.8125226157329659


In [20]:
test = test[cols_to_use].copy()
# y = train['label']

test["text_len"] = test["comment"].str.len()
test["word_count"] = test["comment"].str.split().str.len()

test["question_marks"] = test["comment"].str.count(r"\?")
test["exclamations"] = test["comment"].str.count(r"!")

test[["text_len", "word_count"]]  = test[["text_len", "word_count"]].fillna(0)
test[["question_marks", "exclamations",]] = test[["question_marks", "exclamations"]].fillna(False)

test = test.fillna('Null')
test = add_term_indicator_columns(test,terms=violent_terms,prefix="v")
test = add_term_indicator_columns(test,terms=moral_terms,prefix="m")
test = add_term_indicator_columns(test,terms=group_terms,prefix="g")
test = add_term_indicator_columns(test,terms=personal_attack,prefix="p")

In [30]:
test_probs = modelA.predict_proba(test)
weighted_test_probs = test_probs * final_weights
final_test_preds = np.argmax(weighted_test_probs, axis=1)

# Load sample submission to get correct format
sample_submission = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/Sample.csv')

# Create submission dataframe
submission = pd.DataFrame({
    sample_submission.columns[0]: sample_submission.iloc[:, 0],  # ID column (first column)
    'label': final_test_preds  # Predictions
})

# Verify submission format
print("\nSubmission Preview:")
print(submission.head())
print(f"\nSubmission shape: {submission.shape}")
print(f"Prediction distribution:\n{submission['label'].value_counts().sort_index()}")

# Save submission file
submission.to_csv('submission_hybrid.csv', index=False)
print("\n✓ Submission file saved as 'submission.csv'")

# Verify no missing values
assert submission.isnull().sum().sum() == 0, "WARNING: Submission has missing values!"
print("✓ No missing values in submission")


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Submission Preview:
   ID  label
0   1      2
1   2      2
2   3      0
3   4      0
4   5      2

Submission shape: (102000, 2)
Prediction distribution:
label
0    56719
1     8728
2    33356
3     3197
Name: count, dtype: int64

✓ Submission file saved as 'submission.csv'
✓ No missing values in submission


In [21]:
# probs = modelA.predict_proba(test)

# y_pred = np.argmax(probs, axis=1)

# # Only flip to class 3 if confidence is meaningful
# threshold = 0.2 # tune this

# mask = (probs[:, 3] > threshold)
# y_pred[mask] = 3



# # Load sample submission to get correct format
# sample_submission = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/Sample.csv')

# # Create submission dataframe
# submission = pd.DataFrame({
#     sample_submission.columns[0]: sample_submission.iloc[:, 0],  # ID column (first column)
#     'label': y_pred  # Predictions
# })

# # Verify submission format
# print("\nSubmission Preview:")
# print(submission.head())
# print(f"\nSubmission shape: {submission.shape}")
# print(f"Prediction distribution:\n{submission['label'].value_counts().sort_index()}")

# # Save submission file
# submission.to_csv('submission.csv', index=False)
# print("\n✓ Submission file saved as 'submission.csv'")

# # Verify no missing values
# assert submission.isnull().sum().sum() == 0, "WARNING: Submission has missing values!"
# print("✓ No missing values in submission")


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Submission Preview:
   ID  label
0   1      2
1   2      2
2   3      0
3   4      0
4   5      2

Submission shape: (102000, 2)
Prediction distribution:
label
0    56912
1     7871
2    34207
3     3010
Name: count, dtype: int64

✓ Submission file saved as 'submission.csv'
✓ No missing values in submission
